# Implement Agglomerative hierarchical clustering algorithm using appropriate dataset.
**Computer Laboratory II (417526) – Information Retrieval – Practical 5**

Install once (Linux terminal):
```
pip install scikit-learn scipy pandas matplotlib
```

In [ ]:
import os, re, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
warnings.filterwarnings("ignore")

DATA_DIR = "data"
def get_file(name, url):
    # Return path to data/<name>; download it first if it is not there yet.
    os.makedirs(DATA_DIR, exist_ok=True)
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        print("Downloading", url)
        urllib.request.urlretrieve(url, path)
    return path

In [ ]:
# ---- DATASET (embedded in this notebook) ----
# The data file is stored inside this notebook (compressed), so no download or internet is needed.
# Running this cell writes it to  data/CC_GENERAL.csv
import base64, gzip, os
os.makedirs("data", exist_ok=True)
_DATA = """
...
"""
with open("data/CC_GENERAL.csv", "wb") as f:
    f.write(gzip.decompress(base64.b64decode(_DATA.replace("\n", ""))))
print("data/CC_GENERAL.csv written:", os.path.getsize("data/CC_GENERAL.csv"), "bytes")

## Practical 5 – Agglomerative hierarchical clustering
**Agglomerative (bottom-up) clustering:** start with every point as its own cluster and repeatedly merge the two closest clusters (here **Ward** linkage) until one cluster remains. The history of merges is shown in a **dendrogram**; cutting it with a horizontal line gives the clusters.

**Steps:** load data → drop `CUST_ID`, fill missing values → scale + normalise → PCA to 2 components → dendrogram → clusters for k = 2…6 → **silhouette score** to choose k.
Data set: Credit Card customers (`CC_GENERAL.csv`, 8,950 customers × 17 behavioural features).

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics import silhouette_score
import scipy.cluster.hierarchy as shc

# Step 2: load + clean
X = pd.read_csv(get_file("CC_GENERAL.csv", "https://raw.githubusercontent.com/rohitdileep/Credit-Card-Segmentation-using-Kmeans/master/CC_GENERAL.csv"))
print(X.shape)
X = X.drop("CUST_ID", axis=1)
print("Missing values per column:\n", X.isna().sum()[X.isna().sum() > 0])
X = X.fillna(X.median())          # (the manual's fillna(method='ffill') is deprecated in recent pandas)

# Step 3: preprocessing
X_scaled = StandardScaler().fit_transform(X)
X_normalized = normalize(X_scaled)

# Step 4: dimensionality reduction
pca = PCA(n_components=2)
X_principal = pd.DataFrame(pca.fit_transform(X_normalized), columns=["P1", "P2"])
print("Variance explained by 2 components:", pca.explained_variance_ratio_.sum().round(3))
X_principal.head()

In [ ]:
# Step 5: dendrogram (the last 30 merges are shown so the plot stays readable)
Z = shc.linkage(X_principal, method="ward")
plt.figure(figsize=(9, 6))
shc.dendrogram(Z, truncate_mode="lastp", p=30, show_leaf_counts=True)
plt.axhline(y=Z[-2, 2] + 0.5 * (Z[-1, 2] - Z[-2, 2]), color="k", lw=3)     # cut between the last two merges
plt.title("Visualising the data (Ward dendrogram)"); plt.xlabel("cluster size / sample"); plt.ylabel("distance")
plt.show()
print("The horizontal cut crosses 2 vertical lines -> optimal number of clusters = 2")

In [ ]:
# Step 6: build and visualise the clustering for k = 2 ... 6
ks = [2, 3, 4, 5, 6]
labels = {}
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, k in zip(axes, ks):
    labels[k] = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X_principal)
    ax.scatter(X_principal["P1"], X_principal["P2"], c=labels[k], cmap="rainbow", s=4)
    ax.set_title(f"k = {k}")
plt.tight_layout(); plt.show()

# Step 7: silhouette scores
scores = [silhouette_score(X_principal, labels[k]) for k in ks]
for k, s in zip(ks, scores):
    print(f"k={k}: silhouette = {s:.4f}")
plt.bar(ks, scores); plt.xlabel("Number of clusters"); plt.ylabel("S(i)"); plt.show()
print("Optimal number of clusters (highest silhouette):", ks[int(np.argmax(scores))])